In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_curve, auc

# Suppress common warnings for a cleaner output
warnings.filterwarnings('ignore')

def run_comparative_classification(base_dir="evaluation_results", output_dir="model_comparison_plots"):
    """
    Loads complete datasets and index files, constructs 'correct' and 'misclassified'
    dataframes, performs balancing, runs a comparative analysis of models,
    and plots ROC curves for each stance.
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # --- 1. Load Data and Construct Datasets from Indices ---
    print("Loading datasets and applying classification indices...")

    try:
        # Define paths to all necessary files
        path_wtwt_data = os.path.join(base_dir, 'wtwt_test_processed.csv')
        path_wtwt_correct_idx = os.path.join(base_dir, 'wtwt_correctly_classified_indices.npy')
        path_wtwt_mis_idx = os.path.join(base_dir, 'wtwt_misclassified_indices.npy')

        path_except_data = os.path.join(base_dir, 'except_wtwt_test_processed_mapped_data.csv')
        path_except_correct_idx = os.path.join(base_dir, 'except_wtwt_correctly_classified_indices.npy')
        path_except_mis_idx = os.path.join(base_dir, 'except_wtwt_misclassified_indices.npy')

        # Load the complete dataframes
        df_wtwt = pd.read_csv(path_wtwt_data)
        df_except_wtwt = pd.read_csv(path_except_data)

        # Load the index arrays
        wtwt_correct_indices = np.load(path_wtwt_correct_idx)
        wtwt_mis_indices = np.load(path_wtwt_mis_idx)
        except_correct_indices = np.load(path_except_correct_idx)
        except_mis_indices = np.load(path_except_mis_idx)

    except FileNotFoundError as e:
        print(f"❌ Error: A required file was not found. Please ensure all data and .npy files are in the '{base_dir}' directory.")
        print(f"Missing file: {e.filename}")
        return

    # Use .iloc to select rows based on the loaded indices
    df_wtwt_correct = df_wtwt.iloc[wtwt_correct_indices].copy()
    df_wtwt_mis = df_wtwt.iloc[wtwt_mis_indices].copy()
    df_except_correct = df_except_wtwt.iloc[except_correct_indices].copy()
    df_except_mis = df_except_wtwt.iloc[except_mis_indices].copy()

    # Combine the segmented dataframes
    df_correct_all = pd.concat([df_wtwt_correct, df_except_correct], ignore_index=True)
    df_mis_all = pd.concat([df_wtwt_mis, df_except_mis], ignore_index=True)
    
    # Assign the "Correct" and "Misclassified" labels
    df_correct_all["label"] = "Correct"
    df_mis_all["label"] = "Misclassified"

    print(f"Loaded {len(df_correct_all)} total correct samples and {len(df_mis_all)} total misclassified samples.")

    # --- 2. Perform Undersampling ---
    print("Performing stratified undersampling...")
    misclassified_stance_counts = df_mis_all['stance'].value_counts()

    if misclassified_stance_counts.sum() == 0:
        print("❌ Error: No misclassified examples found. Cannot proceed with analysis.")
        return

    df_correct_sampled = (
        df_correct_all.groupby('stance')
        .apply(lambda x: x.sample(
            n=min(len(x), misclassified_stance_counts.get(x.name, 0)),
            random_state=42
        ))
        .reset_index(drop=True)
    )

    # Combine to create the final, balanced dataset for modeling
    df_balanced = pd.concat([df_correct_sampled, df_mis_all], ignore_index=True)
    df_balanced["label_bin"] = df_balanced["label"].map({"Correct": 1, "Misclassified": 0})

    print(f"Balanced dataset created with {len(df_correct_sampled)} correct and {len(df_mis_all)} misclassified samples.\n")

    # --- 3. Define Models and Feature Columns ---
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'SVM': SVC(kernel='rbf', probability=True, random_state=42),
        'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    }

    # Define feature columns automatically, excluding identifiers
    feature_cols = [
        c for c in df_balanced.columns
        if c not in ['target', 'text', 'stance', 'label', 'label_bin', 'dataset', 'topic', 'split', 'index']
        and pd.api.types.is_numeric_dtype(df_balanced[c])
    ]
    stances = [("All_Stances", None), ("AGAINST", 0), ("FAVOR", 1), ("NONE", 2)]
    print(f"Found {len(feature_cols)} features for modeling.")


    # --- 4. Iterate, Train, and Evaluate --- (This section is unchanged)
    for stance_name, stance_val in stances:
        print(f"\n{'='*70}")
        print(f"Processing Stance: {stance_name}")
        print(f"{'='*70}")

        df_stance = df_balanced.copy()
        if stance_val is not None:
            df_stance = df_balanced[df_balanced["stance"] == stance_val]

        if df_stance.empty or len(df_stance['label_bin'].unique()) < 2:
            print(f"Skipping {stance_name} due to insufficient data or only one class present.")
            continue

        X = df_stance[feature_cols]
        y = df_stance["label_bin"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y
        )

        imputer = SimpleImputer(strategy='median')
        X_train_imputed = imputer.fit_transform(X_train)
        X_test_imputed = imputer.transform(X_test)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_imputed)
        X_test_scaled = scaler.transform(X_test_imputed)

        plt.figure(figsize=(8, 7))

        for model_name, model in models.items():
            print(f"\n--- {model_name} ---")
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            print(classification_report(y_test, y_pred, target_names=['Misclassified', 'Correct'], digits=4))

            # Feature importance logic (unchanged)
            if hasattr(model, 'coef_'): # For LR and linear SVM
                importance_df = pd.DataFrame({
                    'feature': feature_cols,
                    'coefficient': model.coef_[0]
                }).assign(importance=lambda df: np.abs(df['coefficient']))
                top_features = importance_df.sort_values(by='importance', ascending=False).head(10)
                print("\nTop 10 Most Important Features:")
                print(top_features[['feature', 'coefficient']])
            elif hasattr(model, 'feature_importances_'): # For XGBoost
                importance_df = pd.DataFrame({
                    'feature': feature_cols,
                    'importance': model.feature_importances_
                })
                top_features = importance_df.sort_values(by='importance', ascending=False).head(10)
                print("\nTop 10 Most Important Features:")
                print(top_features)

            # ROC Curve Logic
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f'{model_name} (AUC = {roc_auc:.2f})')

        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves for Stance: {stance_name}')
        plt.legend(loc="lower right")
        plt.grid(True)

        plot_path = os.path.join(output_dir, f"roc_curves_{stance_name}.png")
        plt.savefig(plot_path, dpi=300)
        plt.close() # Prevents plots from displaying in a non-interactive environment
        print(f"\nSaved ROC curve plot to: {plot_path}")

if __name__ == '__main__':
    sns.set_theme(style="whitegrid")
    
    # Run the analysis. The script will look for files in the "evaluation_results" directory.
    run_comparative_classification()

Loading datasets and applying classification indices...
Loaded 9769 total correct samples and 758 total misclassified samples.
Performing stratified undersampling...
Balanced dataset created with 758 correct and 758 misclassified samples.

Found 44 features for modeling.

Processing Stance: All_Stances

--- Logistic Regression ---
               precision    recall  f1-score   support

Misclassified     0.4867    0.4803    0.4834       152
      Correct     0.4870    0.4934    0.4902       152

     accuracy                         0.4868       304
    macro avg     0.4868    0.4868    0.4868       304
 weighted avg     0.4868    0.4868    0.4868       304


Top 10 Most Important Features:
                feature  coefficient
0            word_count    -0.790277
34        hedge_C_count     0.687097
4      type_token_ratio    -0.607929
37        hedge_I_count    -0.446578
5        hapax_legomena     0.409671
42              I_ratio     0.326927
9            verb_ratio    -0.311238
7   p